# Data Loading

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\yong-chen.how\Desktop\Chen\Nus\Project\fashion-recommender\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the article metadata
current_folder = Path.cwd()
project_root = current_folder.parent.parent.parent
DATA_DIR = project_root / "data" / "raw"
ARTICLES_PATH = os.path.join(DATA_DIR, "articles.csv")
CUSTOMERS_PATH = os.path.join(DATA_DIR, "customers.csv")
TRANSACTIONS_PATH = os.path.join(DATA_DIR, "transactions_train.csv")

# Ensure IDs are strings for safe joins
articles_df = pd.read_csv(ARTICLES_PATH, dtype={"article_id": "string"})
customers = pd.read_csv(CUSTOMERS_PATH, dtype={"customer_id": "string"})
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    dtype={"customer_id": "string", "article_id": "string"},
    parse_dates=["t_dat"],
)

print("articles:", articles_df.shape)
print("customers:", customers.shape)
print("transactions:", transactions.shape)


articles: (105542, 25)
customers: (1371980, 7)
transactions: (31788324, 5)


In [3]:
# Create Check Function
def check_image_exists(article_id):
    # Get the first 3 digit for the subfolder
    subfolder = article_id[:3]

    # Construct the image path
    image_path = os.path.join(DATA_DIR, "images", subfolder, f"{article_id}.jpg")

    return os.path.exists(image_path)

# Scan and Filter
print(f"Original articles count: {len(articles_df)}")
print("Scanning for images...")

# Check if images exist and filter the articles
articles_df["has_image"] = articles_df["article_id"].apply(check_image_exists)

# Filter articles that have images
clean_articles_df = articles_df[articles_df["has_image"]].copy()

# Drop the helper column
clean_articles_df.drop(columns=["has_image"], inplace=True)

print(f"Articles with images: {len(clean_articles_df)}")
print(f"Dropped articles without images: {len(articles_df) - len(clean_articles_df)}")

Original articles count: 105542
Scanning for images...
Articles with images: 105100
Dropped articles without images: 442


# Top-Bottom Filtering

In [4]:
# Ensure IDs are propperly formatted strings
clean_articles_df["article_id"] = clean_articles_df["article_id"].apply(lambda x: f"{int(x):010d}")

# Determine Gender
is_ladies_index = clean_articles_df['index_group_name'].isin(['Ladieswear', 'Divided'])
is_ladies_dept = clean_articles_df['department_name'].str.contains('Ladies|Women', case=False, na=False)
is_mens_index = clean_articles_df['index_group_name'] == 'Menswear'
is_mens_dept = clean_articles_df['department_name'].str.contains('Mens|Men', case=False, na=False)

print("Articles with Ladieswear index:", is_ladies_index.sum())
print("Articles with Ladieswear department:", is_ladies_dept.sum())
print("Articles with Menswear index:", is_mens_index.sum())
print("Articles with Menswear department:", is_mens_dept.sum())

# Combine the rules
gender_conditions = [
    (is_ladies_index | is_ladies_dept),
    (is_mens_index | is_mens_dept)
]
gender_choices = ['Ladieswear', 'Menswear']

# Apply the logic
clean_articles_df['broad_gender']= np.select(gender_conditions, gender_choices, default='other')

# Determine Type ('top' vs 'bottom')
type_conditions = [
    clean_articles_df['product_group_name'] == 'Garment Upper body',
    clean_articles_df['product_group_name'] == 'Garment Lower body'
]
type_choices = ['top', 'bottom']
clean_articles_df['broad_type'] = np.select(type_conditions, type_choices, default='other')

# 4. Combine them
clean_articles_df['combined_category'] = clean_articles_df['broad_gender'] + ' ' + clean_articles_df['broad_type']

# 5. Filter and Format
valid_items = clean_articles_df[
    (clean_articles_df['broad_gender'] != 'other') & 
    (clean_articles_df['broad_type'] != 'other')
].copy()

category_edges = valid_items[['article_id', 'combined_category']].copy()
category_edges.columns = ['source', 'target']
category_edges['relation'] = 'has_category'

print(f"Generated {len(category_edges)} 'has_category' edges.")
print("\nCategory Distribution:")
print(category_edges['target'].value_counts())

Articles with Ladieswear index: 54636
Articles with Ladieswear department: 1893
Articles with Menswear index: 12504
Articles with Menswear department: 1753
Generated 41389 'has_category' edges.

Category Distribution:
target
Ladieswear top       21872
Ladieswear bottom     9463
Menswear top          7358
Menswear bottom       2696
Name: count, dtype: int64


# Best Matches Based on Transaction

In [5]:
# Create a Category Lookup Dictionary
# Maps '0123456789' -> 'womens top', 'mens bottom', etc.
category_dict = dict(zip(category_edges['source'], category_edges['target']))

print("Mapping categories to transactions...")
# Apply the mapping. If an item isn't a top or bottom, it becomes NaN
transactions['category'] = transactions['article_id'].map(category_dict)

# Drop transactions for accessories, shoes, or unrecognized items
trans_filtered = transactions.dropna(subset=['category']).copy()

# Separate Tops and Bottoms
trans_tops = trans_filtered[trans_filtered['category'].str.contains('top')].copy()
trans_bottoms = trans_filtered[trans_filtered['category'].str.contains('bottom')].copy()

# Extract just the gender word (e.g., split 'womens top' to get 'womens')
trans_tops['gender'] = trans_tops['category'].str.split(' ').str[0]
trans_bottoms['gender'] = trans_bottoms['category'].str.split(' ').str[0]

print("Finding co-occurrences (Same customer, same date)...")
# Merge to find items bought together
merged = pd.merge(
    trans_tops[['customer_id', 't_dat', 'article_id', 'gender']],
    trans_bottoms[['customer_id', 't_dat', 'article_id', 'gender']],
    on=['customer_id', 't_dat'],
    suffixes=('_top', '_bottom')
)

# Enforce Gender Matching
# CRITICAL: Only keep pairs where both items are for the same gender
valid_pairs = merged[merged['gender_top'] == merged['gender_bottom']].copy()

print("Counting frequency of matching outfits...")
# Count the frequencies 
pair_counts = valid_pairs.groupby(
    ['article_id_top', 'article_id_bottom']
).size().reset_index(name='count')

# Sort to find the most frequently bought together pairs
pair_counts = pair_counts.sort_values(['article_id_top', 'count'], ascending=[True, False])

# --- 6. Avoid the K-Core Trap! ---
# Keep the top 5 bottoms for each top, rather than just the single best 1
best_matches = pair_counts.groupby('article_id_top').head(5)

# --- 7. Format for the Knowledge Graph ---
match_edges = best_matches[['article_id_top', 'article_id_bottom', 'count']].copy()
match_edges.columns = ['source', 'target', 'weight']
match_edges['relation'] = 'best_matches_with'

print(f"Generated {len(match_edges)} highly accurate 'best_matches_with' edges.")
display(match_edges.head(10))

Mapping categories to transactions...
Finding co-occurrences (Same customer, same date)...
Counting frequency of matching outfits...
Generated 132316 highly accurate 'best_matches_with' edges.


,source,target,weight,relation
1442,0108775015,0706016002,97,best_matches_with
1441,0108775015,0706016001,91,best_matches_with
373,0108775015,0562245001,74,best_matches_with
9,0108775015,0179123001,71,best_matches_with
1284,0108775015,0688432001,70,best_matches_with
3109,0108775044,0695545001,75,best_matches_with
3188,0108775044,0706016002,72,best_matches_with
3113,0108775044,0695545007,69,best_matches_with
3187,0108775044,0706016001,55,best_matches_with
2304,0108775044,0562245050,44,best_matches_with


# Define Occasion & Seasons

In [6]:
# Extract Clean the Description Columns
print("Extracting Descriptions.....")
clean_articles_df['clean_desc'] = clean_articles_df['prod_name'].fillna('') + " - " + clean_articles_df['detail_desc'].fillna('')

print("Loading SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Define the Target Categories ---
occasion_prompts = {
    'Sport': 'Activewear, gym, running, workout, sports, training, breathable, athletic, yoga',
    'Formal': 'Formal wear, business attire, office, suit, tailored, blazer, elegant dress, evening wear',
    'Casual': 'Casual everyday wear, relaxed, comfortable, denim, t-shirt, basic, lounge',
    'Party': 'Party wear, going out, clubbing, sequin, festive, glamour, cocktail dress, nightlife'
}
occasion_labels = list(occasion_prompts.keys())
occasion_vectors = model.encode(list(occasion_prompts.values()))

# Generate Product Embeddings
descriptions_list = clean_articles_df['clean_desc'].tolist()

print(f"Embedding {len(descriptions_list)} product descriptions...")
product_vectors = model.encode(descriptions_list, show_progress_bar=True)

# Calculate Cosine Similarity
print('Calclating similarity scores...')
similarity_matrix = cosine_similarity(product_vectors, occasion_vectors)

# Assign the best matching occasion to each product
best_match_indices = np.argmax(similarity_matrix, axis=1)

# Map the index back to the actual word
clean_articles_df['assigned_occasion'] = [occasion_labels[idx] for idx in best_match_indices]

clean_articles_df['confidence_score'] = np.max(similarity_matrix, axis=1)

print("\n--- Classification Complete! ---")
display(clean_articles_df[['article_id', 'clean_desc', 'assigned_occasion', 'confidence_score']].head(10))



Extracting Descriptions.....
Loading SentenceTransformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1087.19it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 105100 product descriptions...


Batches: 100%|██████████| 3285/3285 [10:00<00:00,  5.47it/s]


Calclating similarity scores...

--- Classification Complete! ---


,article_id,clean_desc,assigned_occasion,confidence_score
0,0108775015,Strap top - Jersey top with narrow shoulder st...,Casual,0.306843
1,0108775044,Strap top - Jersey top with narrow shoulder st...,Casual,0.306843
2,0108775051,Strap top (1) - Jersey top with narrow shoulde...,Casual,0.309744
3,0110065001,OP T-shirt (Idro) - Microfibre T-shirt bra wit...,Casual,0.330353
4,0110065002,OP T-shirt (Idro) - Microfibre T-shirt bra wit...,Casual,0.330353
5,0110065011,OP T-shirt (Idro) - Microfibre T-shirt bra wit...,Casual,0.330353
6,0111565001,20 den 1p Stockings - Semi shiny nylon stockin...,Casual,0.344047
7,0111565003,20 den 1p Stockings - Semi shiny nylon stockin...,Casual,0.344047
8,0111586001,Shape Up 30 den 1p Tights - Tights with built-...,Casual,0.216854
9,0111593001,Support 40 den 1p Tights - Semi shiny tights t...,Sport,0.381892


In [8]:
import joblib

joblib.dump(clean_articles_df, 'clean_articles_df.joblib')

['clean_articles_df.joblib']